# Customer Churn Analysis & Retention Strategy

## Objective:
Analyze customer behavior to identify key drivers of churn and provide actionable business recommendations.

In [ ]:
# Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load Data
df = pd.read_csv('C:/Path/Raw/telco_customer_churn.csv', na_values=['',' ', 'NA', 'null'])

## Raw Data Preview

In [ ]:
df.head(10)

In [ ]:
df.info()

## Data Cleaning
- Renaming Columns for Consistency
- Convertion to binary variables (Yes/No → 0/1) and categorical datatype
- Handling missing values
- Checking for Duplicates and Inconsistencies

In [ ]:
# Renaming Columns
df.rename(columns={"customerID": "CustomerID", "gender":"Gender", "tenure":"Tenure"}, inplace=True)

# Binary Mapping
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
binary_map = {"Yes": 1, "No": 0}
for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Convert Categorical Data
categorical_cols = [
    "Gender", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection","TechSupport", "StreamingTV", 
    "StreamingMovies", "Contract", "PaymentMethod"
]
for col in categorical_cols:
    df[col] = df[col].astype("category")

# Fix TotalCharges
## Conversion
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
## Assumption (Total Charges are zero due to tenure being 0)
print(df[df["TotalCharges"].isnull()]["Tenure"].unique())
## Fix
df.loc[df["Tenure"] == 0, "TotalCharges"] = 0
## Fix (in case assumption not true in all cases)
df = df.dropna(subset=["TotalCharges"])

## Data Verification

In [ ]:
# Checking Duplicate Values
df.duplicated().sum()

In [ ]:
# CHecking Unique Values to ensure consistency
for col in df.columns:
    print(col, df[col].unique())

In [ ]:
# Validating Data Distribution
df.describe()

## Feature Engineering

In [ ]:
# Tenure Group
df["TenureGroup"] = pd.cut(
    df["Tenure"],
    bins=[-1, 12, 24, 48, 60, 100],
    labels=["0-1yr", "1-2yr", "2-4yr", "4-5yr", "5+yr"]
)

In [ ]:
# Total Services Count
services = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
df["TotalServices"] = df[services].apply(lambda x: (x == "Yes").sum(), axis=1)

## Add internet + phone separately
df["TotalServices"] += (df["InternetService"] != "No").astype(int)
df["TotalServices"] += (df["PhoneService"] == 1).astype(int)

In [ ]:
# Avg Charges
df["AvgCharges"] = df["TotalCharges"] / df["Tenure"]
df["AvgCharges"] = df["AvgCharges"].fillna(0)

### Validating Data Consistency for MonthlyCharges

In [ ]:
(df["AvgCharges"] - df["MonthlyCharges"]).sum()

Since the difference in amount is not significant, it shows consistency in billing and controls and hence MonthlyCharges can be dropped to reduce redundancy.

### Dropping AvgCharges After Analysis to Reduce Redundancy

In [ ]:
df.drop(columns=["AvgCharges"], inplace=True)

## Clean Data Preview

In [ ]:
df.head(10)

In [ ]:
df.info()

## Analysis

### Overview
This section analyzes overall churn trends and key patterns in customer behavior.

In [ ]:
# Churn Rate
churn_rate = df["Churn"].mean() * 100
print(f"Churn Rate: {churn_rate:.2f}%")

# Churn Distribution
plt.figure(figsize=(4,4))
sns.countplot(x="Churn", data=df)
plt.title("Customer Churn Distribution")
plt.show()

Approximately 26% of customers have churned, indicating a significant customer retention challenge. This highlights the need to identify key drivers of churn and implement targeted retention strategies.

### Customer Behaviour

In [ ]:
# Churn By Tenure Group
plt.figure(figsize=(8,5))
sns.barplot(x="TenureGroup", y="Churn", data=df)
plt.title("Churn By Tenure Group")
plt.show()

Churn rates are highest among customers in the initial tenure groups and decrease steadily as tenure increases, reinforcing the importance of onboarding and early engagement strategies.

In [ ]:
# Churn by Contract Type
plt.figure(figsize=(8,5))
sns.barplot(x="Contract", y="Churn", data=df)
plt.title("Churn Rate by Contract Type")
plt.show()

Customers on month-to-month contracts exhibit the highest churn rates, likely due to the lack of long-term commitment. In contrast, customers on longer-term contracts show significantly lower churn, highlighting the effectiveness of contract-based retention.

### Services Impact
This section evaluates how customer engagement with services influences churn likelihood.

In [ ]:
# Churn By Total Services
plt.figure(figsize=(8,5))
sns.barplot(x="TotalServices", y="Churn", data=df)
plt.title("Churn By Total Services")
plt.show()

Churn decreases as the number of subscribed services increases, indicating that customer engagement plays a critical role in retention. Customers with fewer services are significantly more likely to churn, highlighting an opportunity for service bundling strategies.

In [ ]:
# Churn by Individual Services
services = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

for col in services:
    plt.figure(figsize=(4,3))
    sns.barplot(x=col, y="Churn", data=df)
    plt.title(f"Churn Rate by {col}")
    plt.show()

Customers who utilize support and after-sales services (such as tech support, online security, and backup services) exhibit lower churn rates, suggesting that these services enhance customer satisfaction and increase retention. In contrast, entertainment services like streaming TV and movies show minimal impact on churn, indicating limited differentiation in a highly competitive segment.  
This highlights the importance of value-added support services over entertainment add-ons in driving customer retention.

### Revenue Analysis

In [ ]:
# Churn by Monthly Charges
plt.figure(figsize=(8,5))
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

Customers with higher monthly charges tend to churn more, suggesting pricing sensitivity and potential dissatisfaction with perceived value.

In [ ]:
# Churn by Total Charges
plt.figure(figsize=(8,5))
sns.boxplot(x="Churn", y="TotalCharges", data=df)
plt.title("Total Charges vs Churn")
plt.show()

Customers with lower total charges show higher churn rates, likely due to shorter tenure, reinforcing the importance of early customer retention.

In [ ]:
# Correlation Heatmap
cols = [
    "Tenure",
    "MonthlyCharges",
    "TotalCharges",
    "TotalServices",
    "Churn"
]
plt.figure(figsize=(10,6))
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

The correlation analysis indicates that tenure has the strongest relationship with churn, showing a moderate negative correlation. This suggests that customers who remain longer with the company are significantly less likely to churn, emphasizing the importance of early customer retention.

The observed relationships for monthly and total charges further reinforce the patterns identified in the earlier analysis, particularly regarding pricing sensitivity and customer lifecycle effects.

Interestingly, total services show little to no correlation with churn, suggesting that while service adoption may influence behavior in other analyses, it is not a strong linear predictor of churn on its own. While total services show weak correlation in the heatmap, above analysis using categorical comparisons reveals its impact on churn, suggesting that correlation alone may not fully capture complex customer behavior.

## Final Analysis & Conclusion

This analysis aimed to identify the key factors influencing customer churn and uncover actionable insights to improve customer retention.

The overall churn rate of approximately 26% highlights a significant retention challenge. Through exploratory analysis, several key patterns emerged across customer behavior, contract types, service usage, and pricing.

Customer tenure was identified as one of the strongest indicators of churn, with newer customers significantly more likely to leave. Churn rates decrease steadily as tenure increases, emphasizing the importance of customer retention during the early stages of the lifecycle.

Contract type also plays a critical role, with customers on month-to-month contracts exhibiting substantially higher churn compared to those on longer-term agreements. This suggests that long-term commitments are strongly associated with improved retention.

Service engagement further differentiates customer behavior. Customers with fewer subscribed services tend to churn more frequently, while those utilizing additional services demonstrate greater retention. This indicates that increased engagement enhances customer stickiness and perceived value.

Pricing-related factors also influence churn. Higher monthly charges are associated with increased churn, suggesting price sensitivity among customers. The observed relationships for monthly and total charges further reinforce the patterns identified in the earlier analysis, particularly regarding pricing sensitivity and customer lifecycle effects.

Correlation analysis supports these findings, highlighting tenure as the most influential factor negatively associated with churn, while pricing shows a positive relationship. Behavioral factors such as engagement and lifecycle stage appear to have a stronger impact on churn compared to demographic characteristics.

Overall, the analysis suggests that churn is primarily driven by a combination of low customer engagement, lack of long-term commitment, early-stage lifecycle vulnerability, and pricing sensitivity.

### Business Recommendations

- Encourage long-term contracts through targeted incentives and discounts  
- Strengthen onboarding and early engagement strategies to reduce initial churn  
- Promote service bundling to increase customer engagement and retention  
- Review pricing strategies for high-cost segments to improve perceived value  
- Implement targeted retention campaigns for high-risk customer groups  

By focusing on these areas, the business can reduce churn, improve customer lifetime value, and enhance overall customer satisfaction.